In [1]:
print("Antas")

Antas


In [2]:
# ABC Optimization & Data Setup
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report

# Load the original CLEANED text dataset (Not the encoded train.csv)
df_tab = pd.read_csv('30k_ppd_CTGAN_cleaned.csv')

# --- ABC OPTIMIZATION ---
# Fix the rare anomaly in the Gravida column
df_tab['Gravida'] = df_tab['Gravida'].replace({'Multigravida+D1191': 'Multigravida'})

# Define which columns are numbers and which are text (Categories)
target = 'Labelling'
numerical_cols = ['Age', 'Gestational Age', 'Number of sons', 'Number of daughters',
                  'Total Number of Children', 'Scalling']

# Every other column except numericals and target is a categorical column
categorical_cols = [col for col in df_tab.columns if col not in numerical_cols and col != target]

print(f"Numerical Columns ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical Columns ({len(categorical_cols)}): {categorical_cols}")
print("\nABC Optimization Complete! No rare anomalies remain.")

Numerical Columns (6): ['Age', 'Gestational Age', 'Number of sons', 'Number of daughters', 'Total Number of Children', 'Scalling']
Categorical Columns (20): ['Gravida', 'Female Education', 'Husband Education', 'Working Status', 'Physical Health', 'Previous Miscarriage', 'Sufficient Money for Basic Needs', 'Current Appereance Acceptance', 'Family System', 'Male Gender Preference', 'Relationship with Mother in-law', 'Little interest or pleasure in doing things', 'Feeling down, depressed, or hopeless', 'Trouble falling or staying sleep or sleeping too much', 'Feeling tired or having little energy', 'Poor appetite or overeating', 'Feeling badabout yourself that you are failure or have let yourself or your family down', 'Trouble concentrating on things, such as reading the newspaper or watching television', 'Moving or speaking so slowly that other people could have Noticed.', 'Thoughts that you would be better off dead, or of hurting yourself']

ABC Optimization Complete! No rare anomalies 

In [3]:
# BLOCK 2: TabTransformer Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib

# 1. Scale the Numerical Columns
scaler = StandardScaler()
df_tab[numerical_cols] = scaler.fit_transform(df_tab[numerical_cols])
joblib.dump(scaler, 'tab_scaler.pkl')

# 2. Label Encode the Categorical Columns
# We also need to save the "vocabulary size" (how many unique categories exist in each column)
# because the Multi-Head Attention layers need to know exactly how many "embeddings" to build.
vocab_sizes = []
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_tab[col] = le.fit_transform(df_tab[col].astype(str))
    vocab_sizes.append(len(le.classes_))
    encoders[col] = le

joblib.dump(encoders, 'tab_encoders.pkl')

# 3. Train/Test Split (80/20)
X = df_tab.drop(target, axis=1)
y = df_tab[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Data scaled, encoded, and split perfectly for the TabTransformer!")
print(f"Vocabulary sizes for Attention Layers: {vocab_sizes}")

Data scaled, encoded, and split perfectly for the TabTransformer!
Vocabulary sizes for Attention Layers: [2, 6, 6, 2, 2, 2, 3, 2, 2, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4]


In [5]:
# Formatting Data for TabTransformer
import tensorflow as tf

# Keras functional API requires inputs as a dictionary or list
def prepare_transformer_inputs(X, cat_cols, num_cols):
    # Dictionary for categorical columns
    inputs = {col: X[col].values for col in cat_cols}
    # Single array for numerical columns
    inputs['numeric_input'] = X[num_cols].values
    return inputs

X_train_tf = prepare_transformer_inputs(X_train, categorical_cols, numerical_cols)
X_test_tf = prepare_transformer_inputs(X_test, categorical_cols, numerical_cols)

print("Data successfully formatted into Keras tensor inputs!")

Data successfully formatted into Keras tensor inputs!


In [8]:
# BLOCK 4 (UPGRADED): High-Capacity TabTransformer
from tensorflow.keras.layers import Input, Embedding, Concatenate, Dense, Flatten, MultiHeadAttention, LayerNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("--- Initializing High-Capacity TabTransformer ---")

# 1. Define Inputs & Thicker Embeddings
cat_inputs = []
cat_embeddings = []
embed_dim = 32 # UPGRADED: Doubled the mathematical vectors

for i, col in enumerate(categorical_cols):
    inp = Input(shape=(1,), name=col)
    cat_inputs.append(inp)

    vocab_size = vocab_sizes[i]
    emb = Embedding(input_dim=vocab_size, output_dim=embed_dim)(inp)
    cat_embeddings.append(emb)

numeric_input = Input(shape=(len(numerical_cols),), name='numeric_input')

# 2. Upgraded Transformer Block
x_cat = Concatenate(axis=1)(cat_embeddings)

# UPGRADED: 8 Attention Heads instead of 4
attention_out = MultiHeadAttention(num_heads=8, key_dim=embed_dim)(x_cat, x_cat)
x_cat = LayerNormalization()(x_cat + attention_out)
x_cat = Flatten()(x_cat)

x = Concatenate()([x_cat, numeric_input])

# 3. Deeper MLP Head
x = Dense(128, activation='relu')(x) # UPGRADED: Thicker dense layer
x = Dropout(0.2)(x) # Reduced dropout slightly to allow more data flow
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(32, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

tab_model = Model(inputs=cat_inputs + [numeric_input], outputs=output)

# Slower, more careful initial learning rate
optimizer = Adam(learning_rate=0.001)
tab_model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# 4. Dynamic Learning Rate & Patient Early Stopping
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001, verbose=1)

# Wait 20 epochs before giving up completely
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

print("Training started...")
history = tab_model.fit(
    X_train_tf, y_train,
    validation_split=0.2,
    epochs=100, # Increased max epochs
    batch_size=32, # UPGRADED: Smaller batch size for better generalization
    callbacks=[early_stop, lr_scheduler],
    verbose=0
)

# 5. Evaluate
tab_probs = tab_model.predict(X_test_tf)
tab_preds = (tab_probs > 0.5).astype(int).flatten()

tab_acc = accuracy_score(y_test, tab_preds)

tab_model.save('tabtransformer_optimized.h5')

--- Initializing High-Capacity TabTransformer ---
Training started...

Epoch 15: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.

Epoch 20: ReduceLROnPlateau reducing learning rate to 4.0000001899898055e-05.

Epoch 25: ReduceLROnPlateau reducing learning rate to 1e-05.
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step


In [14]:
print("\nClassification Report:")
print(classification_report(y_test, tab_preds))


Classification Report:


              precision    recall  f1-score   support

           0       0.97      0.96      0.97      2140
           1       0.98      0.98      0.98      3860

    accuracy                           0.98      6000
   macro avg       0.98      0.97      0.98      6000
weighted avg       0.98      0.98      0.98      6000

